In [ ]:
!pip install groq --quiet

import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.2 MB/s eta 0:00:00


In [ ]:
from groq import Groq
API_KEY='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'
client= Groq(api_key=API_KEY)
MODEL='llama-3.1-8b-instant'
print(f"Groq client configured with model:{MODEL}")
print("Make sure API_KEY is replacecd with your actual Key")

Groq client configured with model:llama-3.1-8b-instant
Make sure API_KEY is replacecd with your actual Key


In [ ]:
def ask_llm(user_message, system_message="You are a helpful assistant",temperature=0.7,max_tokens=500):
    """
        user_message   : The question or task fro the model
        system_message : Instructions about role and behaviuor
        temperature    : 0.0=deterministic 1.0=creative index
        max_tokens     : decides the token to be used for the answer
     """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content":user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content
test_response=ask_llm("What do you expect a Data engineering + gen AI specalist to do?")
test_response2=ask_llm("News Letter to follow the latest technologies in 2026")
print(test_response)
print(f"\n{test_response2}")

A Data Engineering + Gen AI Specialist is a highly specialized role that combines expertise in data engineering, artificial intelligence, and machine learning. Here are some key responsibilities and expectations for this role:

**Data Engineering (40-50% of the role):**

1. **Design and implement data pipelines**: Create efficient, scalable, and fault-tolerant data pipelines to collect, transform, and load data from various sources into data warehouses or lakes.
2. **Data storage and management**: Design and implement data storage solutions using relational databases, NoSQL databases, data warehouses, or big data stores like HDFS, S3, or Azure Data Lake.
3. **Data processing and analytics**: Develop data processing workflows using tools like Apache Spark, Hadoop, or Flink, and integrate them with data analytics frameworks like Apache Beam or AWS Glue.
4. **Data quality and governance**: Ensure data accuracy, completeness, and consistency by implementing data validation, data profiling,

In [ ]:
response_etl=ask_llm(
    "IN 3 bullet points, explain how the Medallion Architecture"
    "(Bronze,Silver,Gold Layers) relates to ETL pipelines",
    system_message="You are a senior data engineering insturctor."
                   "Be concise and practical."
)
print("Medallion + ETL connection:")
print(response_etl)
print()
print('Token explanation')
print("The model above used approximately", len(response_etl.split())*1.3,'tokens,')

Medallion + ETL connection:
Here are 3 bullet points explaining how the Medallion Architecture (Bronze, Silver, Gold Layers) relates to ETL pipelines:

• **Bronze Layer (Raw Data Ingest)**: This layer corresponds to the **Ingest** phase of ETL, where raw data is collected from various sources (e.g., databases, APIs, files) and loaded into a data warehouse or lake. The Bronze layer is responsible for handling the high-volume, high-velocity, and high-variety data streams.

• **Silver Layer (Raw Data Processing)**: This layer aligns with the **Transform** phase of ETL, where the raw data is processed and transformed into a standardized format. The Silver layer involves data cleansing, data quality checks, data normalization, and data aggregation to create a processed dataset that is ready for analysis.

• **Gold Layer (Curated Data)**: This layer corresponds to the **Load** phase of ETL, where the processed data from the Silver layer is loaded into a data mart or a reporting database, mak

In [ ]:
zero_shot_response=ask_llm(
    "Extract the city name from this address: "
    "456 Brigade Road, Bangalore 560025, Karnataka,India"
)
print('Zero-Shot Result:')
print(zero_shot_response)
print()
ambiguous_response = ask_llm("Clean this data: ramesh kumar,45000,mumbai")
print('Ambiguous Result:')
print(ambiguous_response)

Zero-Shot Result:
The city name extracted from the address is: Bangalore

Ambiguous Result:
The data appears to be a record of a person's name, salary, and location. 

Here's the cleaned data with proper formatting and potential issues addressed:

- Name: Ramesh Kumar
- Salary: 45,000
- Location: Mumbai

If we assume the data is to be stored in a structured format, such as a spreadsheet or a database, the cleaned data might look like this:

| Name | Salary | Location |
|------|--------|----------|
| Ramesh Kumar | 45000 | Mumbai |

Note: In the original data, the salary value is in a simple text format, which might lead to potential issues if the data is used for numerical calculations. It's recommended to store numerical values in a format that allows for mathematical operations, such as a numeric field in a database or a spreadsheet.


In [ ]:
few_shot_prompt="""
Convert employee text to JSON. Here are examples:
Input:RAMESH KUMAR,45000,mumbai
Output:{"name":"RAMESH KUMAR","salary":45000,"city":"mumbai"}
Input:ramesh kumar,45000,Delhi
Output:{"name":"ramesh kumar","salary":45000,"city":"Delhi"}
Now convert this:
Input: ANANYA DAS, 38000 , kolkata
Output:"""
few_shot_response=ask_llm(few_shot_prompt,temperature=0.0)
print('Few-Shot Result:')
print(few_shot_response)
print()
try:
  parsed=json.loads(few_shot_prompt.strip())
  print('Successfully parsed as JSON')
  print(f'Name:{parsed['name']},Salary:{parsed['salary']},City:{parsed["city"]}')
except json.JSONDecodeError:
  print('Parsing failed-model added extra text')
  print('Solution: add explicit instructions in the system prompt')

Few-Shot Result:
To convert the employee text to JSON, we can use the following Python code:

```python
import json

def convert_to_json(employee_text):
    # Split the input string into individual values
    values = employee_text.split(',')

    # Create a dictionary with the employee details
    employee = {
        "name": values[0].strip(),
        "salary": int(values[1].strip()),
        "city": values[2].strip()
    }

    # Convert the dictionary to JSON
    json_output = json.dumps(employee, indent=4)

    return json_output

# Test the function
employee_text = "ANANYA DAS, 38000 , kolkata"
print(convert_to_json(employee_text))
```

When you run this code, it will output:

```json
{
    "name": "ANANYA DAS",
    "salary": 38000,
    "city": "kolkata"
}
```

This code works by splitting the input string into individual values using the comma as a delimiter. It then creates a dictionary with the employee details, stripping any leading or trailing whitespace from each value. Fin

In [ ]:
question = "Review this python code and identify any issues:\n"\
           "df['Revenue'] = df['qty]* df['price]\n"\
           "result = df.groupby('dept).sum()"

generic_response = ask_llm(question, temperature=0.2)
print('Without Role Prompting:')
print(generic_response[:300], '...')
print()

role_response = ask_llm(
    question,
    system_message = "You are a senior Data Engineer with 10 years of production"
                     "experience. Review code criticality for production readiness."
                     "data types issues, and potential failures at scale.",
    temperature=0.2,
    max_tokens=500
)
print("With role Prompting(Senior Data Engineer):")
print(role_response[:400], '...')
print()
print('Notice: role prompting produces more technical, actionable feedaback')

Without Role Prompting:
There are several issues with the provided Python code:

1.  **Syntax Error**: There's a missing closing parenthesis in the line where you're multiplying 'qty' and 'price'. It should be `df['Revenue'] = df['qty'] * df['price']`.

2.  **Typo**: In the line where you're grouping the DataFrame by 'dept ...

With role Prompting(Senior Data Engineer):
**Code Review**

The provided Python code appears to be a snippet from a larger data analysis or data engineering project. Here's a review of the code, highlighting potential issues and suggestions for improvement:

```python
# Potential issue: missing closing parenthesis in the multiplication operation
df['Revenue'] = df['qty'] * df['price']

# Potential issue: missing closing parenthesis in the  ...

Notice: role prompting produces more technical, actionable feedaback


In [ ]:
prompt = "Give me one creative name for a data analytics startup."
print("=== Temperature Experiment ===")
for temp in [0.0,0.5,1.0]:
  response = ask_llm(prompt, temperature=temp)
  print(f"Temperature {temp}: {response.strip()}")
  time.sleep(1)
print()
print('Observation:')
print(" temperature = 0.0 -> same or very similar answer every run(deterministic) ")
print(" temperature = 0.5 -> same or some variation")
print(" temperature = 0.0 -> more creative/varied, sometimes surprising")
print()
print('Rule for data engineering tasks: use temperature= 0.0 or 0.1')
print('You need CONSISTENT, PARSABLE output - not creative variation')


=== Temperature Experiment ===
Temperature 0.0: Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys the idea of a startup that helps businesses navigate complex data landscapes and uncover hidden patterns and trends.
Temperature 0.5: Here's a creative name for a data analytics startup:

**"Nexixa"**

This name suggests a connection or nexus between data and insights, which is at the heart of what a data analytics startup does. It also has a modern and tech-savvy sound to it, which could appeal to potential customers and investors.
Temperature 1.0: Here's a creative name for a data analytics startup: "Nexus Insights"

Nexus refers to the connection or link between different pieces of data, which is at the heart of what data analytics does. Adding "Insights" conveys the idea that the startup provides valuable, actionable info

In [ ]:
invoice_text = "Invoice #2024-001 from TECHWORLD SOLUTIONS dated 15th January 2024. Amount Rs. 45,000 for Laptop"

weak_response = ask_llm(
    f"Clean this invoice data: {invoice_text}",
    temperature=0.3
)
print('WEAK PROMPT OUTPUT:')
print(weak_response)
print()

try:
  json.loads(weak_response)
  print('Successfully parsed as JSON')
except json.JSONDecodeError:
  print("Parsing failed-model added extra text")
  print('Solution: add explicit instructions in the system prompt')

strong_response = """You are a data extraction speacalist for an accounting pipeline.
                extract invoice data and return only a valid json object.
                DO not include any explanation, preamble, or markdown formatting.
                Retuen ONLY the JSON, nothing else.

                JSON Schema(use null for missing values):
                {"invoice_id:" string, "vendor_name": string(Title Case),
                 "amount": number (no currency symbols),
                 "currency": string(default INR)}"""

print('STRONG PROMPT OUTPUT:')
print(strong_response)
try:
  parsed=json.loads(strong_response)
  print('Successfully parsed as JSON')
  print('Invoice ID:{parsed["invoice_id"]},Vendor name:{parsed["Vendor_name"]}')
except json.JSONDecodeError:
  print('Parsing failed for strong prompt.')
  print('Response was:', strong_response)

WEAK PROMPT OUTPUT:
Here's the cleaned invoice data:

**Invoice Details:**

- **Invoice Number:** 2024-001
- **Date:** 15th January 2024
- **Company Name:** TECHWORLD SOLUTIONS
- **Item Purchased:** Laptop
- **Amount:** Rs. 45,000

Let me know if you need any further assistance.

Parsing failed-model added extra text
Solution: add explicit instructions in the system prompt
STRONG PROMPT OUTPUT:
You are a data extraction speacalist for an accounting pipeline.
                extract invoice data and return only a valid json object.
                DO not include any explanation, preamble, or markdown formatting.
                Retuen ONLY the JSON, nothing else.
                
                JSON Schema(use null for missing values):
                {"invoice_id:" string, "vendor_name": string(Title Case),
                 "amount": number (no currency symbols),
                 "currency": string(default INR)}
Parsing failed for strong prompt.
Response was: You are a data extraction